# Required installations

In [ ]:
!pip install -q transformers accelerate datasets huggingface_hub

# Required imports

In [ ]:
import time
import torch
from datasets import Dataset
from huggingface_hub import login
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

# HugggingFace connection and model details

In [ ]:
api = HfApi()
model_id = "oki692/LFM2.5-1.2B-Base"
info = api.model_info(model_id)

print("Used model:", info.modelId)
print("Created at (release date):", info.created_at)
print("Last updated:", info.lastModified)


Used model: oki692/LFM2.5-1.2B-Base
Created at (release date): 2026-03-02 03:23:29+00:00
Last updated: 2026-03-02 03:23:29+00:00


# Model building and running

To use the llm the following should be loaded:


*   ***model:*** The llm to be used.
*   ***tokenizer:*** The specific tokeniziner used for this llm to tokenize inputs of the model which are as texts in our case.



In [ ]:
# Load the base model and tokenizer
model_id = "oki692/LFM2.5-1.2B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

# 📋 Explanation of arguments in run_model

- **prompt**  
  The text you give the model as input. This is what the model will respond to.

- **max_tokens (default=100)**  
  The maximum number of new tokens (words/pieces of text) the model is allowed to generate.  
  Example: `max_tokens=50` limits the output length.

- **tokenizer(prompt, return_tensors="pt")**  
  Converts the input text into numerical form (tokens) that the model can understand.  
  `return_tensors="pt"` means the result is returned as PyTorch tensors.

- **.to(model.device)**  
  Moves the input tensors to the same device as the model (CPU or GPU) so they can be processed.

- **inputs.pop("token_type_ids", None)**  
  Removes the `token_type_ids` key if it exists. Some models don’t need it, so this avoids errors.

- **torch.no_grad()**  
  Tells PyTorch not to track gradients during generation. This makes inference faster and uses less memory.

- **model.generate(...)**  
  The function that actually produces text from the model. Key arguments inside:
  - `**inputs` → passes the tokenized prompt to the model.  
  - `max_new_tokens=max_tokens` → sets how many tokens to generate.  
  - `do_sample=False` → disables randomness; the model picks the most likely next token (deterministic output).  
  - `pad_token_id=tokenizer.eos_token_id` → ensures padding uses the end‑of‑sequence token, avoiding errors.

- **tokenizer.decode(outputs[0], skip_special_tokens=True)**  
  Converts the generated tokens back into human‑readable text.  
  `skip_special_tokens=True` removes things like `<eos>` or `<pad>` from the final output.


In [ ]:
def run_model(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    inputs.pop("token_type_ids", None)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test cases (10 data points)

# 📋 Why each reason matters

- **Complex arithmetic** → Large or unusual calculations often trip models, so this checks math reliability.
- **Non‑English multilingual** → Tests direct translation to non‑English languages, where models are weaker.
- **Obscure factual recall** → Rare historical facts expose gaps in memory and knowledge coverage.
- **Ancient literature knowledge** → Forces the model to handle old texts outside common training data.
- **Creative form constraint** → Checks if the model can follow strict rules (like rhyme and rhythm).
- **Ambiguous sentiment with mixed cues** → Sentiment analysis gets harder when emotions are mixed.
- **Pattern recognition** → Sequences like Fibonacci test logical continuation skills.
- **Less common geography with dual capitals** → Many models miss countries with unusual capital setups.
- **Logic puzzle with positional constraints** → Requires step‑by‑step reasoning, not just recall.
- **Cross‑lingual translation** → Tests pivot translation (e.g., Spanish → latin), where errors often creep in.


In [ ]:
# 📊 Test cases chosen to maximize incorrect outputs
test_cases = [
    {"input": "What is the square root of 12345?",
     "expected_output": "≈111.11",
     "reason": "Complex arithmetic"},

    {"input": "Translate 'The mitochondria is the powerhouse of the cell' into Japanese.",
     "expected_output": "ミトコンドリアは細胞の力の源です",
     "reason": "Non‑English multilingual"},

    {"input": "Who won the Nobel Prize in Physics in 1933?",
     "expected_output": "Erwin Schrödinger and Paul Dirac",
     "reason": "Obscure factual recall"},

    {"input": "Summarize the plot of 'The Epic of Gilgamesh' in 1 sentence.",
     "expected_output": "Gilgamesh seeks immortality after Enkidu dies...",
     "reason": "Ancient literature knowledge"},

    {"input": "Write a limerick about machine learning.",
     "expected_output": "Proper limerick rhyme and rhythm",
     "reason": "Creative form constraint"},

    {"input": "I guess I’m happy, in a sad sort of way — is the tone positive, negative, or neutral?",
     "expected_output": "Neutral",
     "reason": "Ambiguous sentiment with mixed cues"},

    {"input": "Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8, 13, ...",
     "expected_output": "21, 34, 55",
     "reason": "Pattern recognition"},

    {"input": "What is the capital of Eswatini?",
     "expected_output": "Mbabane (administrative), Lobamba (royal/legislative)",
     "reason": "Less common geography with dual capitals"},

    {"input": "Three people (A, B, and C) are sitting in a row. A is not next to B. C is not at either end. Who is in the middle?Show the reason for your answer.",
     "expected_output": "C",
     "reason": "Logic puzzle with positional constraints"},

    {"input": "Translate 'Hola, ¿cómo estás?' into Latin.",
     "expected_output": "Salve, quid agis?",
     "reason": "Cross‑lingual translation into ancient language"}

]

# Running on tests cases

In [ ]:
# Run the model on each case and collect outputs
%%time
examples = []
for case in test_cases:
    output = run_model(case["input"])
    examples.append({
        "input": case["input"],
        "expected_output": case["expected_output"],
        "model_output": output,
        "reason": case["reason"]
    })

CPU times: user 5min 54s, sys: 510 ms, total: 5min 54s
Wall time: 6min 12s


In [ ]:
def pretty_print_example(example: dict):
    print("📝 Test Case")
    print(f"Input:            {example['input']}")
    print(f"Expected Output:  {example['expected_output']}")
    print(f"Model Output:     {example['model_output']}")
    print(f"Reason:           {example['reason']}")
    print("-" * 60)

# Usage: loop through your examples list
for ex in examples:
    pretty_print_example(ex)

📝 Test Case
Input:            What is the square root of 12345?
Expected Output:  ≈111.11
Model Output:     What is the square root of 12345?  
12345 is not a perfect square.  
#### 111.8  

A rectangular garden has a length that is 3 times its width. If the perimeter of the garden is 160 meters, what is the area of the garden?  
Let the width be $ x $, then the length is $ 3x $.  
Perimeter: $ 2(x + 3x) = 8x = 160 $ → $ x = 20 $.  
Area: $ 
Reason:           Complex arithmetic
------------------------------------------------------------
📝 Test Case
Input:            Translate 'The mitochondria is the powerhouse of the cell' into Japanese.
Expected Output:  ミトコンドリアは細胞の力の源です
Model Output:     Translate 'The mitochondria is the powerhouse of the cell' into Japanese.

The mitochondria is the powerhouse of the cell.
Reason:           Non‑English multilingual
------------------------------------------------------------
📝 Test Case
Input:            Who won the Nobel Prize in Physics in 1933

# Building and uploading the final dataset

The dataset now contains the model outputs while previously it contained the expected output. These two outputs can be now compared and the dataset is ready to be uploaded.

In [ ]:
# Build dataset
dataset = Dataset.from_list(examples)

# Push to Hugging Face Hub (replace with your username/repo name)
dataset.push_to_hub("usernam/blindspot-LFM2.5-1.2B-Base")